# Imports

In [1]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import json
import cv2

# > for gemini API call
from google import genai

# > for langchain google genai chat model
from langchain_google_genai import ChatGoogleGenerativeAI

# > raster engine
from raster_engine_module import raster_engine

# > for making messages
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# > for langchain agent
from langchain.agents import create_agent

# > for structured output tool strategy
from langchain.agents.structured_output import ToolStrategy

# > for initializing chat model
from langchain.chat_models import init_chat_model

# > for image to base64 conversion
import io
import base64

# > for getting API key securely
import getpass


c:\Users\soura\miniconda3\envs\EOAgent\Lib\site-packages\pyogrio\core.py:36: RuntimeWarning: Could not detect PROJ data files. Set PROJ_LIB environment variable to the correct path.
  _init_proj_data()


# All sensor paths

In [2]:
rp=raster_engine()

In [ ]:

sentinel_path=rp.make_valid_path(r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\sentinel 2\sentinel 2\sentinel2_aois_all_bands_OrigRes")

landsat_path=rp.make_valid_path(r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\landsat\bands_from_B2_to_B6")

lis3_path=rp.make_valid_path(r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\bhoonidhi\Lis3\croped_bands_tifs")

lis4_path=rp.make_valid_path(r"C:\Users\soura\Desktop\my_files\IIIT\GPT\New code\images for testing VLMs\bhoonidhi\lis4\all_cropped_bands")

# formatting functions

In [ ]:
class formatting_functions:
    def __init__(self):
        pass
    
    def format_vegetation_data(
        self,
        data_dict
        ):
        formatted_data_dict=dict()
        
        def format_cell_dict(cell_dict):
            formatted_cell_dict=dict()
            
            def format_area_coverage_info(area_coverage_info_dict):
                
                formatted_area_coverage_info_dict=dict()
                    
                for k,v in area_coverage_info_dict.items():
                        formatted_area_coverage_info_dict[k]=round(v,9)
                        
                return formatted_area_coverage_info_dict
            
            def format_ratio_coverage_info(ratio_coverage_info_dict):
                
                formatted_ratio_coverage_info_dict=dict()
                
                for k,v in ratio_coverage_info_dict.items():
                        formatted_ratio_coverage_info_dict[k]=round(v,2)
                        
                return formatted_ratio_coverage_info_dict
                    
            
            for k,v in cell_dict.items():
                
                if k=='total area covered by cell in km2':
                    formatted_cell_dict['total area covered by cell in km2']=round(v,9)
                    
                elif k=='area covered by cell':
                    formatted_cell_dict['area covered by cell km2']=round(v,9)
                
                
                elif k == "vegetation class coverage area in km2":
                    formatted_cell_dict[k]=v.copy()
                    
                    formatted_cell_dict[k]['coverage_info']=format_area_coverage_info(
                        v['coverage_info']
                        )
                    
                elif k == "vegetation class coverage %":
                    formatted_cell_dict[k]=v.copy()
                    
                    formatted_cell_dict[k]['coverage_info']=format_ratio_coverage_info(
                        v['coverage_info']
                        )
                
                else: formatted_cell_dict[k]=v.copy()
                
            return formatted_cell_dict
                
        
        cell_map={
            'cell_1':"upper left",
            'cell_2':"upper middle",
            'cell_3':"upper right",
            'cell_4':"middle left",
            'cell_5':"center",
            'cell_6':"middle right",
            'cell_7':"lower left",
            'cell_8':"lower middle",
            'cell_9':"lower right"}
        
        if len(data_dict)>0:
            for cell, data in data_dict.items():
                formatted_data_dict[cell_map[cell]] = format_cell_dict(data)
            
        return formatted_data_dict
    
    def format_built_data(
        self,
        data_dict
        ):
        formatted_data_dict=dict()
        
        def format_cell_dict(cell_dict):
            formatted_cell_dict=dict()
            
            def format_area_coverage_info(area_coverage_info_dict):
                
                formatted_area_coverage_info_dict=dict()
                    
                for k,v in area_coverage_info_dict.items():
                        formatted_area_coverage_info_dict[k]=round(v,9)
                        
                return formatted_area_coverage_info_dict
            
            def format_ratio_coverage_info(ratio_coverage_info_dict):
                
                formatted_ratio_coverage_info_dict=dict()
                
                for k,v in ratio_coverage_info_dict.items():
                        formatted_ratio_coverage_info_dict[k]=round(v,2)
                        
                return formatted_ratio_coverage_info_dict
                    
            
            for k,v in cell_dict.items():
                
                
                if k == "built-up class coverage %":
                    
                    formatted_cell_dict[k]={}
                    
                    for k2,v2 in v.items():
                        if k2=="coverage":
                            formatted_cell_dict[k]["coverage_info"]=v2.copy()
                            
                        else:
                            formatted_cell_dict[k][k2]=v2
                            
                    
                    formatted_cell_dict[k]['coverage_info']=format_ratio_coverage_info(
                        v['coverage']
                        )
                    
                else:
                    formatted_cell_dict[k]=v
                
            return formatted_cell_dict
                
        
        cell_map={
            'cell_1':"upper left",
            'cell_2':"upper middle",
            'cell_3':"upper right",
            'cell_4':"middle left",
            'cell_5':"center",
            'cell_6':"middle right",
            'cell_7':"lower left",
            'cell_8':"lower middle",
            'cell_9':"lower right"}
        
        if len(data_dict)>0:
            for cell, data in data_dict.items():
                formatted_data_dict[cell_map[cell]] = format_cell_dict(data)
            
        return formatted_data_dict
    
    def format_overall_data(
        self,
        data_dict
        ):
        
        formatted_data_dict=dict()
        
        if len(data_dict)>0:
            for k,v in data_dict.items():
                if k=='total area in km2':
                    formatted_data_dict[k]=round(v,9)
                else:
                    formatted_data_dict[k]=v.copy()
                
        return formatted_data_dict
    
    def format_water_data(
        self,
        data_dict
        ):
        formatted_data_dict=dict()
        
        if len(data_dict)>0:
            for k,v in data_dict.items():
                
                formatted_v=dict()
                formatted_v["bounding box"]=v["bounding box"].copy()

                formatted_v["area in m2"]=round(v["area in m2"],3)
                
                formatted_data_dict[f"water body {k}"]=formatted_v
            
        return formatted_data_dict

# model instruction and user input

In [8]:
model_instructions="""\
You are an expert in remote sensing and geo-spatial analysis tasked with generating high-quality answers for questions about satellite imagery.

---

# Instruction

Your task is to analyze the provided satellite image along with its metadata and answer the given question.

You will be provided with:
1. A satellite image (JPG)
2. Metadata including:
   - Vegetation coverage statistics (3x3 grid)
   - Built-up area statistics (3x3 grid)
   - Water bodies information (bounding boxes and areas)
   - Overall land cover statistics
   - Location information
   - Image bounds
3. A question about the image

---

# Answering Rules (IMPORTANT)

1. **Answer ONLY the question. Do NOT explain your reasoning.**
2. **Be concise and factual.**
3. **Do NOT add background context, calculations, or commentary.**
4. **If the answer is numeric or a percentage, return ONLY the value (with unit if applicable).**
5. **Use the provided metadata as the primary source of truth.**
6. **Do NOT restate the question in the answer.**\
"""


In [9]:



user_text_input="""\
vegetation_coverage_statistics = {vegetation_data}     # JSON object with vegetation coverage statistics for each cell
built_up_area_statistics = {built_up_data}    # JSON object with built-up area statistics for each cell
water_bodies_information = {water_data}  # JSON object with water body centroids and areas
overall_land_cover_statistics = {overall_stats_data}    # Json object with overall land cover statistics in percentage and km2  
location_information = {location_data}       # JSON object with location information
image_bounds = {bounds_data}    # Json object containing geo coordinated for image bounding box

**question** : {question}\
"""

# Output schema

**reference code**:

```
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


product_review_schema = {
    "type": "object",
    "description": "Analysis of a product review.",
    "properties": {
        "rating": {
            "type": ["integer", "null"],
            "description": "The rating of the product (1-5)",
            "minimum": 1,
            "maximum": 5
        },
        "sentiment": {
            "type": "string",
            "enum": ["positive", "negative"],
            "description": "The sentiment of the review"
        },
        "key_points": {
            "type": "array",
            "items": {"type": "string"},
            "description": "The key points of the review"
        }
    },
    "required": ["sentiment", "key_points"]
}

agent = create_agent(
    model="gpt-5",
    tools=tools,
    response_format=ToolStrategy(product_review_schema)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# {'rating': 5, 'sentiment': 'positive', 'key_points': ['fast shipping', 'expensive']}
```

In [10]:
question_ans_schema= {
    "type": "object",
    "description": "Answer of the given question",
    "properties": {
        "answer": {
            "type": ["string"],
            "description": "your answer for the given question."
        }
    },
    "required": ["answer"]
}

# make human_message with image

In [11]:
def make_human_message(
    formatted_user_text_input:str,
    jpg_img
):
    # * jpg_img is a PIL Image object - convert to base64
    img_byte_arr = io.BytesIO()
    jpg_img.save(img_byte_arr, format='JPEG')
    img_byte_arr.seek(0)
    image_data = base64.standard_b64encode(img_byte_arr.getvalue()).decode("utf-8")

    # * List of standard content blocks
    human_message = HumanMessage(content_blocks=[
        {
            "type": "text", "text": formatted_user_text_input
        },
        {
            "type": "image_url", "image_url": {
                                            "url": f"data:image/jpeg;base64,{image_data}"
                                            },
        },
    ])
    
    return human_message

# function for resizing image

In [12]:
from PIL import Image
import math

def resize_image_to_limit(high_res_img, max_pixels=33177600):
    
    import math
    
    width, height = high_res_img.size
    current_pixels = width * height

    # If already within limit, just save and return
    if current_pixels <= max_pixels:
        # print("Image under limit — no resizing needed.")
        return high_res_img

    # Compute scale factor
    scale_factor = math.sqrt(max_pixels / current_pixels)

    new_width = int(width * scale_factor)
    new_height = int(height * scale_factor)

    print(
        f"Resizing image from ({width}x{height}={current_pixels} px) "
        f"→ ({new_width}x{new_height}={new_width*new_height} px)"
    )

    # Resize with high quality
    resized_img = high_res_img.resize((new_width, new_height), Image.LANCZOS)
    return resized_img


# function for extracting json data

In [13]:

def get_data(
    bands_dir_path:str,
    sensor:str
):
    
    formatter=formatting_functions()
    info_file_key_words=[]
        
    if sensor!='lis4':
        info_file_key_words=[
            "water",
            "built",
            "veg",
            "overall",
            "location",
            "bounds"
            ]
    else:
        info_file_key_words=[
            "water",
            "veg",
            "overall",
            "location",
            "bounds"
            ]
    
    paths_dict=dict()
    
    for file_key_word in info_file_key_words:
        
        paths_dict[f"{file_key_word}_info_path"] = [
            os.path.join(
                bands_dir_path,
                file_name) for file_name in os.listdir(bands_dir_path)
            if (file_key_word in file_name) and (file_name.endswith(".json"))
            ][0]
    
    
    data_dict=dict()
    
    for path_name, path in paths_dict.items():
        
        file_key_word=path_name.split("_")[0]
        data_name_key=f"{file_key_word}_data"
        
        try:
            with open(path, "r", encoding="utf-8") as f:
                
                json_data=json.load(f)
                # data_dict[data_name_key]=json_data
                
                
                if file_key_word=="water":
                    data_dict[data_name_key]=formatter.format_water_data(json_data)
                    
                elif file_key_word=="built":
                    data_dict[data_name_key]=formatter.format_built_data(json_data)
                    
                elif file_key_word=="veg":
                    data_dict[data_name_key]=formatter.format_vegetation_data(json_data)
                    
                elif file_key_word=="overall":
                    data_dict[data_name_key]=formatter.format_overall_data(json_data)
                
                else:
                    data_dict[data_name_key]=json_data
                
        except FileNotFoundError:
            print(f"{path} file not found!")
    
    # * open the image
    image_path=[
            os.path.join(
                bands_dir_path,
                file_name) for file_name in os.listdir(bands_dir_path)
            if file_name.endswith(".jpg")
            ][0]
    
    
    img = Image.open(image_path)
    
    # *  resize the image if it exceeds the limit
    resized_img=resize_image_to_limit(img)
    
            
    return data_dict,resized_img

# run for each sensor

In [ ]:
def run_for_sensor(
    sensor:str,
    parent_dir_path:str,
    questions,
    agent
):
    
    def save_in_json(
            content:dict,
            path:str,
            file_name:str
            ):
            """Saves a dictionary as json file.

            Args:
                content (dict): Dictionary containing the content
                path (str): The directory in which json file will be saved, for eg. "C/abc/.../sentinel/all_bands"
                file_name (str): Name of the json file, for eg. "water_info.json"
            """
            
            with open(os.path.join(path,file_name), "w") as f:
                json.dump(content, f)
                
    global model_instructions
    global user_text_input

    sensor_ques_ans_dict=dict()
    unsaved_aoi_ques_ans_dict=dict()
        
    bands_dir_names=os.listdir(parent_dir_path)
    
    for i,band_dir_name in enumerate(bands_dir_names[12:], start=12):
    
        print(i," : ",band_dir_name)
        bands_dir_path=os.path.join(
            parent_dir_path,
            band_dir_name
        )
        
        saved_ans_json_data={}
        
        # * if generated_answers.json already exists, then open it
        if "generated_answers.json" in os.listdir(bands_dir_path):
            
            # import warnings
            # warnings.warn("generated_answers.json already exists, hence will be updated!")
            print("\n!!!!!!!!! generated_answers.json already exists, hence will be updated !!!!!!!!!\n")
            
        
            # * open answers json file
            
            ans_file_name="generated_answers.json"
            ans_file_path=os.path.join(
                bands_dir_path,
                ans_file_name
            )
        
            with open(ans_file_path, "r", encoding="utf-8") as f:
                saved_ans_json_data=json.load(f)
                
        
        # * if generated_answers.json not present, then make it
        else:
            save_in_json(
                content=saved_ans_json_data,
                path=bands_dir_path,
                file_name="generated_answers.json"
            )
            
            
        json_data,jpg_img=get_data(
            bands_dir_path=bands_dir_path,
            sensor=sensor
        )
        
        # * run for only unsaved questions
        unsaved_questions=questions
        updated_aoi_ques_ans_dict=None
        
        
        for q in unsaved_questions:
            print(q)
            formatted_user_text_input=user_text_input.format(
                question=q,
                vegetation_data = json_data['veg_data'],
                built_up_data = json_data['built_data'] if sensor!='lis4' else None,
                water_data = json_data['water_data'],
                overall_stats_data = json_data['overall_data'],
                location_data = json_data['location_data'],
                bounds_data = json_data['bounds_data'])
        
            system_message = SystemMessage(model_instructions)
            human_message = make_human_message(
                formatted_user_text_input=formatted_user_text_input,
                jpg_img=jpg_img
                )
                
                
            messages = [
                system_message,
                human_message
                ]
            
            return messages

            
            agent_response=agent.invoke({"messages":messages})
            
            unsaved_aoi_ques_ans_dict[q]=agent_response["structured_response"]
            
        
            updated_aoi_ques_ans_dict = unsaved_aoi_ques_ans_dict.copy()
            
            # * add new question, answers from unsaved_aoi_ques_ans_dict to updated_aoi_ques_ans_dict
            if len(saved_ans_json_data)!=0:
                
                updated_aoi_ques_ans_dict = saved_ans_json_data.copy()
                
                for new_q, new_a in unsaved_aoi_ques_ans_dict.items():
                    updated_aoi_ques_ans_dict[new_q]=new_a
            
            
            # * save as json in the directory
            save_in_json(
                content=updated_aoi_ques_ans_dict,
                path=bands_dir_path,
                file_name="generated_answers.json"
            )
        
        sensor_ques_ans_dict[band_dir_name]=updated_aoi_ques_ans_dict
        
        

    return sensor_ques_ans_dict

        

    

# questions

In [15]:
questions_dict = {
    "caption": [
        "What does the image show? Summarize the content of the image.",
        "List 3 most prominent landscape features (e.g., river, forest patch, urban area)."
  ],
    "vegetation": [
        "What is the overall state of vegetation in the image?",
        "What is % of the area covered by vegetation?",
        "What is % of the area covered by dense vegetation?",
        "Which area in the image has the most vegetation?",
        "Provide a binary (yes/no) answer: is there stressed/dry vegetation visible? Give evidence."
  ],
    "water": [
        "Are there any water bodies in the image? (yes/no) If yes, list how many and their types (river, pond, reservoir).",
        "What is the total area covered by water (number + units)?",
        "What is the area of the largest water body?",
        "Describe the morphology of the main water bodies (linear, lentic, fragmented).",
        "Give the overall summary of water bodies in the image"
  ],
    "built_up": [
        "Is there evidence of human settlement or built-up structures? (yes/no)",
        "What % of area is covered by built-up structures?",
        "Which part of the image has the most built-up area?",
        "Give an overall summary of built-up structures in the image."
  ],
    "objects_and_features": [
        "List detected man-made objects (roads, bridges, stadiums, large buildings).",
        "List detected natural objects (large trees, bare soil, wetlands)."
  ],
    "overall": [
        "Provide an integrated summary describing vegetation, water and built-up, including key numbers (percentages/areas)."
        # "Provide an combined summary describing vegetation, water and built-up, including key numbers (percentages/areas)."
  ]
}

In [16]:
all_ques_list=list()
for ques_list in questions_dict.values():
    all_ques_list.extend(ques_list)

In [17]:
all_ques_list

['What does the image show? Summarize the content of the image.',
 'List 3 most prominent landscape features (e.g., river, forest patch, urban area).',
 'What is the overall state of vegetation in the image?',
 'What is % of the area covered by vegetation?',
 'What is % of the area covered by dense vegetation?',
 'Which area in the image has the most vegetation?',
 'Provide a binary (yes/no) answer: is there stressed/dry vegetation visible? Give evidence.',
 'Are there any water bodies in the image? (yes/no) If yes, list how many and their types (river, pond, reservoir).',
 'What is the total area covered by water (number + units)?',
 'What is the area of the largest water body?',
 'Describe the morphology of the main water bodies (linear, lentic, fragmented).',
 'Give the overall summary of water bodies in the image',
 'Is there evidence of human settlement or built-up structures? (yes/no)',
 'What % of area is covered by built-up structures?',
 'Which part of the image has the most b

In [ ]:
numeric_ques_index=[3,4,5,8,9,13,14]
numeric_ques = [all_ques_list[i] for i in numeric_ques_index]

text_ques=[all_ques_list[i] for i in range(len(all_ques_list)) if i not in numeric_ques_index]

In [19]:
numeric_ques

['What is % of the area covered by vegetation?',
 'What is % of the area covered by dense vegetation?',
 'Which area in the image has the most vegetation?',
 'What is the total area covered by water (number + units)?',
 'What is the area of the largest water body?',
 'What % of area is covered by built-up structures?',
 'Which part of the image has the most built-up area?']

In [20]:
set(text_ques),len(text_ques)

({'Are there any water bodies in the image? (yes/no) If yes, list how many and their types (river, pond, reservoir).',
  'Describe the morphology of the main water bodies (linear, lentic, fragmented).',
  'Give an overall summary of built-up structures in the image.',
  'Give the overall summary of water bodies in the image',
  'Is there evidence of human settlement or built-up structures? (yes/no)',
  'List 3 most prominent landscape features (e.g., river, forest patch, urban area).',
  'List detected man-made objects (roads, bridges, stadiums, large buildings).',
  'List detected natural objects (large trees, bare soil, wetlands).',
  'Provide a binary (yes/no) answer: is there stressed/dry vegetation visible? Give evidence.',
  'Provide an integrated summary describing vegetation, water and built-up, including key numbers (percentages/areas).',
  'What does the image show? Summarize the content of the image.',
  'What is the overall state of vegetation in the image?'},
 12)

In [21]:
lis4_text_ques=list(set(text_ques) - set(['Is there evidence of human settlement or built-up structures? (yes/no)','Give an overall summary of built-up structures in the image.']))

lis4_text_ques,len(lis4_text_ques)

(['What is the overall state of vegetation in the image?',
  'Provide a binary (yes/no) answer: is there stressed/dry vegetation visible? Give evidence.',
  'Describe the morphology of the main water bodies (linear, lentic, fragmented).',
  'List detected natural objects (large trees, bare soil, wetlands).',
  'What does the image show? Summarize the content of the image.',
  'Give the overall summary of water bodies in the image',
  'Provide an integrated summary describing vegetation, water and built-up, including key numbers (percentages/areas).',
  'Are there any water bodies in the image? (yes/no) If yes, list how many and their types (river, pond, reservoir).',
  'List 3 most prominent landscape features (e.g., river, forest patch, urban area).',
  'List detected man-made objects (roads, bridges, stadiums, large buildings).'],
 10)

# Initialize model

In [ ]:

os.environ["GROQ_API_KEY"]="GROQ_API_KEY"

# * initializing model
model=init_chat_model("meta-llama/llama-4-scout-17b-16e-instruct",model_provider="groq")

# make agent

In [ ]:
agent = create_agent(
                     model,
                     response_format=ToolStrategy(question_ans_schema)
                     )

# Running for sensors

## landsat

In [ ]:
ques_ans=run_for_sensor(
    sensor="landsat",
    parent_dir_path=landsat_path,
    questions=text_ques,
    agent=agent
)

## lis3

In [ ]:
ques_ans=run_for_sensor(
    sensor="lis3",
    parent_dir_path=lis3_path,
    questions=text_ques,
    agent=agent
)

## sentinel

In [ ]:
ques_ans=run_for_sensor(
    sensor="sentinel",
    parent_dir_path=sentinel_path,
    questions=text_ques,
    agent=agent
)

## lis4

In [ ]:
ques_ans=run_for_sensor(
    sensor="lis4",
    parent_dir_path=lis4_path,
    questions=lis4_text_ques,
    agent=agent
)